In [2]:
!pip install pyspark

                                              0.0/317.3 MB ? eta -:--:--
                                              0.3/317.3 MB 7.0 MB/s eta 0:00:46
                                             1.0/317.3 MB 10.2 MB/s eta 0:00:31
                                             1.6/317.3 MB 11.1 MB/s eta 0:00:29
                                             2.2/317.3 MB 11.6 MB/s eta 0:00:28
                                             2.8/317.3 MB 12.0 MB/s eta 0:00:27
                                             3.5/317.3 MB 12.3 MB/s eta 0:00:26
                                             4.1/317.3 MB 12.4 MB/s eta 0:00:26
                                             4.7/317.3 MB 12.4 MB/s eta 0:00:26
                                             5.3/317.3 MB 12.6 MB/s eta 0:00:25
                                             6.0/317.3 MB 12.8 MB/s eta 0:00:25
                                             6.6/317.3 MB 12.7 MB/s eta 0:00:25
                                             7.

In [1]:
import os
import sys
from config import *

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
collection_uri = MONGO_CONN_URI + "student_erp_copy"
spark_connector = "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0"

In [2]:
from IPython.core.interactiveshell import InteractiveShell


In [3]:
InteractiveShell.ast_node_interactivity = "all"
import pyspark
from pyspark.sql.functions import *
from pyspark.sql import SparkSession, SQLContext, functions as F
from pyspark import SparkContext, SparkConf
from config import *

spark = (SparkSession
         .builder
         .master("local")
         .appName("BDA_Assignment")
         .config("spark.driver.memory", "15g")
         .config("spark.mongodb.read.connection.uri", collection_uri)
         .config("spark.mongodb.write.connection.uri", collection_uri)
         .config("spark.jars.packages", spark_connector)
         .getOrCreate())



In [4]:
import time
execution_times = []

In [5]:
df_students = (spark.read
      .format("mongodb")
      .option("uri", collection_uri)
      .option("database", "student_erp_copy")
      .option("collection", "students")
      .load()
)   

In [6]:
df_departments = (spark.read
      .format("mongodb")
      .option("uri", collection_uri)
      .option("database", "student_erp_copy")
      .option("collection", "departments")
      .load()
)   

In [7]:
df_instructors = (spark.read
      .format("mongodb")
      .option("uri", collection_uri)
      .option("database", "student_erp_copy")
      .option("collection", "instructors")
      .load()
)   

### OPTIMIZED QUERY-1

In [34]:
from pyspark.sql.functions import col, explode
import time

start_time = time.time()

# Step 1: Select only necessary columns before exploding to reduce data load
df_students_selected = df_students.select("student_name", "_id", "enrollments")

# Step 2: Unwind the 'enrollments' array to treat each enrollment as a separate row
df_students_enrollments = df_students_selected.select(
    "student_name", 
    "_id", 
    explode(col("enrollments")).alias("enrollment")
)
# Caching the exploded data
df_students_enrollments.cache()

# Step 3: Push the filter early for the specific course
df_students_in_course = df_students_enrollments.filter(
    col("enrollment.course_id") == "CSE101"
)

# Step 4: Selecting only the relevant columns
df_students_in_course = df_students_in_course.select(
    "student_name", 
    "_id", 
    col("enrollment.course_id"), 
    col("enrollment.course_name")
)
df_students_in_course.show(truncate=False)

# Unpersisting the cached data
df_students_enrollments.unpersist()

end_time = time.time()
print(f"Execution Time: {end_time - start_time} seconds")


DataFrame[student_name: string, _id: string, enrollment: struct<course_id:string,course_name:string,credits:int,category:string,instructor:struct<instructor_id:string,instructor_name:string,email:string,phone:string>,enrolled_semester:int>]

+-------------+-------+---------+---------------+
|student_name |_id    |course_id|course_name    |
+-------------+-------+---------+---------------+
|Vikram Bansal|BT22029|CSE101   |Data Structures|
|Karthik Reddy|BT22041|CSE101   |Data Structures|
|Praveen Yadav|BT22043|CSE101   |Data Structures|
|Neha Kapoor  |BT24044|CSE101   |Data Structures|
|Tanvi Bansal |BT23058|CSE101   |Data Structures|
|Isha Mehta   |BT22064|CSE101   |Data Structures|
|Kiran Gupta  |BT24063|CSE101   |Data Structures|
|Tanya Desai  |BT22079|CSE101   |Data Structures|
|Priti Iyer   |BT22082|CSE101   |Data Structures|
|Tara Kapoor  |BT23100|CSE101   |Data Structures|
+-------------+-------+---------+---------------+



DataFrame[student_name: string, _id: string, enrollment: struct<course_id:string,course_name:string,credits:int,category:string,instructor:struct<instructor_id:string,instructor_name:string,email:string,phone:string>,enrolled_semester:int>]

Execution Time: 0.7925307750701904 seconds


### OPTIMISED QUERY-2

In [29]:
from pyspark.sql.functions import explode, col, count, avg

start_time = time.time()

# Projecting only necessary fields (_id and enrollment.instructor_id, enrollment.course_id)
df_students_filtered = df_students.select("_id",explode("enrollments").alias("enrollment")).select(
    "_id",
    "enrollment.instructor.instructor_id",
    "enrollment.course_id")

# Using projection and filtering
instructor_id = "MEC001"
df_students_instructor = df_students_filtered.filter(col("instructor_id") == instructor_id)

# Group by course_id and counting the number of students per course
df_students_per_course = df_students_instructor.groupBy("course_id").agg(count("_id").alias("num_students"))

# Calculating average number of students enrolled in courses taught by the instructor
df_avg_students = df_students_per_course.agg(avg("num_students").alias("avg_students_enrolled"))

df_avg_students.show()

end_time = time.time()
execution_times.append(end_time - start_time)
print(f"Execution Time: {end_time - start_time} seconds")


+---------------------+
|avg_students_enrolled|
+---------------------+
|   25.307692307692307|
+---------------------+

Execution Time: 1.4191453456878662 seconds


### OPTIMIZED QUERY-5

In [33]:
start_time = time.time()

# Cache frequently used dataframes
df_students.cache()
df_instructors.cache()

# Get list of CSE core courses
cse_core_courses = df_departments.filter(col("_id") == "CSE") \
    .select(explode("courses").alias("course")) \
    .filter(col("course.category") == "Core") \
    .select("course.course_id").rdd.flatMap(lambda x: x).collect()

# Explode enrollments and use broadcast join for instructors
df_exploded_enrollments = df_students.select(col("_id").alias("student_id"), explode(col("enrollments")).alias("enrollment")).cache()

# Perform a broadcast join for smaller dataset
df_instructors_taught_courses = df_exploded_enrollments.join(broadcast(df_instructors),
    df_exploded_enrollments.enrollment.instructor.instructor_id == df_instructors._id)

# Filter by core courses and group by instructor
df_filtered_instructors = df_instructors_taught_courses.filter(col("enrollment.course_id").isin(cse_core_courses))

# Aggregate and filter for instructors who have taught all core courses
instructors_with_all_courses = df_filtered_instructors.groupBy("instructor_name", "email") \
    .agg(collect_set("enrollment.course_id").alias("courses_taught")) \
    .filter(size(col("courses_taught")) == len(cse_core_courses))

instructors_with_all_courses.show(truncate=False)

end_time = time.time()
execution_times.append(end_time - start_time)
print(f"Optimized Execution Time: {end_time - start_time} seconds")

DataFrame[_id: string, admission_year: int, department_id: string, dob: string, email: string, enrollments: array<struct<course_id:string,course_name:string,credits:int,category:string,instructor:struct<instructor_id:string,instructor_name:string,email:string,phone:string>,enrolled_semester:int>>, gender: string, phone: string, student_name: string]

DataFrame[_id: string, courses_taught: array<struct<course_id:string,course_name:string,credits:int,category:string>>, department_id: string, email: string, instructor_name: string, phone: string]

+---------------+--------------------------+------------------------------------------------------------------------+
|instructor_name|email                     |courses_taught                                                          |
+---------------+--------------------------+------------------------------------------------------------------------+
|Dr. Priya Desai|priya.desai@university.edu|[CSE202, CSE303, CSE302, CSE301, CSE201, CSE402, CSE101, CSE401, CSE403]|
+---------------+--------------------------+------------------------------------------------------------------------+

Optimized Execution Time: 2.8340342044830322 seconds
